# R39 · Understand the remaining missed fronts

Choose a **standard CPU runtime**, then **Run all**. This small notebook replays the completed PR123 raw-proposal diagnosis against your accepted R38 labels. It downloads only the pinned scripts, labels and numerical evidence; it needs no GPU, model weights, source photos or ML package installation.

**Main result:** 6 of the 11 misses are confidence-gated, 4 lack a decoded box meeting the fixed IoU criterion, and 1 loses its qualifying box to NMS. An extra NMS pass after clipping removes **8 duplicates and 1 other unmatched output**, preserving every existing match and corner result. The separate 0.05 confidence probe recovers five matches but adds 68 unmatched outputs. Neither experiment automatically changes the selected defaults. Duplicates are already included in unmatched totals.

R38 amendments and earlier PDFs are accepted. Detection remains separate from corner availability. Existing permission to predict hidden corners is preserved; a predicted point does not become observed training truth.


In [ ]:
import hashlib, json, os, sys, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

sys.dont_write_bytecode = True
SOURCE_REVISION = 'a3b76772f40bdb8beb3005015c86d99a4eb6b89f'
FILES = {'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000007.txt': '88fa1a222a2ba7aa7bb453e04711694fd6b2205bc0c861ff69df0e5d95fd2921', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000008.txt': '4e83a14adeaff7e60ee5c3d423cb3338507407d7b92b5f88a6b5befcc773f4e5', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000009.txt': '581b913266547da042b2f45817438066841733faf822fe3d10a9e376b02a1774', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000010.txt': '540d4801b3597abdc6982309cba0424235b89a5471d6f3f8a8ece3f92ce7326d', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000011.txt': '92bac561a1071a1a9769194e289858efe3d67ad32adf02dcb202bfade23eb838', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000053.txt': '40f5da5a1594e35889b4df0bbd4a5ca273f64db722242a30ec51c393d8b8e87e', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000054.txt': 'a141b368e31100930826c3c3b777c066c5050b52759161e2b12cf429bd9d0b20', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000055.txt': '54531ae2addfa6cb3aaf6d9677a0d7d94f2be73017b515a36a61f365e160c936', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000056.txt': '56557ff588c710572df2d4e75812f76566c235187423aa098127d168be696eea', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000057.txt': '1b119bcbbe3985974b954da1940a8914242b61b7d8dc75c049f952a4fac5627f', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000058.txt': '0d7cc2962aa94337a3e5340cda114a23bab0c8e3c590855ebbce1e1ca28c1b0c', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000059.txt': '60e6866fda66ad0d385094c32fd41f19eb451c5a16365b41da77cf5f0e01ade4', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000086.txt': '9d6c732962fae773027d9721b235959463942e08304f0fd844c981933609f1e5', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000087.txt': '17d7537f9626058aca885a2afb4ebc1dfa2d8b89acb818e90c09ecbe1113eab1', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000088.txt': '1cb9dfbf1aa06c1d9d062ccfe2453e262b9e8e76d2f29a6170771e71baae14f0', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000089.txt': 'a7b83d04894e165e80d9ff6bd30b3ed59bdb49aec62afd17ab0179785726eb06', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000090.txt': '576a5ec9cc55e59c28379c0571becbc33a485c402e6921d7390a6c9e8aa9029d', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000146.txt': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000147.txt': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'data/references/sequence-refined-reference-2026-11-18/labels/val/new_000148.txt': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'data/references/sequence-refined-reference-2026-11-18/manifest.json': '0de64be3797dce2e43d4cb98d09620eab969bb95725ed9e75682dcfcd53c7f96', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000007.txt': 'def064435049f2216024200eb45fcd91f3ce044fd767082ff017f38314e78ff1', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000008.txt': 'c099d721a98581bca71fc010b69ce9a321930583b68268de452aab052b6ee7cf', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000009.txt': '034cfe5fd0ada90a3831ab3b6043f644edaad0eb3a3c203b2a6a2dce83390f98', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000010.txt': '1b776a7cba10295a1fb7006121004f32de9e97b1e26e5d39173ca459036ce4c2', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000011.txt': 'e0811d8d991c827ae76523e6ff0b166d766698df94c826b15b82558ba9df59f8', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000053.txt': '57b17aaab7b5737b5195ef1ea80bfd30c21a676c4d2a3cce2db2d916f07bfc4b', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000054.txt': '7df21b71d428028c3224739eb2efe13858032d8832ed89e2adc6ff963938e6ac', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000055.txt': 'ab1a7a39857f58214809dbf467917551597738379f1e181570bf9dd25205e2f7', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000056.txt': 'c8791a0b725cb1b4df4f3d771b3e4834954b1d094afdb4ae3e5f41e8d253470d', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000057.txt': '6b6613daaa78b1ab7ad415e629baeb4a74223be96373c3293a8527808970fe01', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000058.txt': 'd99b48a7d1fdcb943defd5e01f6c233223c0586ea95e5aa7d21d9ea851b20aa5', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000059.txt': 'b627ce416f6ce09138673a5b6638b80336361caba71e2a52bfb2547bdc128c29', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000086.txt': '7ac900b13abf7e8aaa0db4c5a178251f346c25a3d581a1689815e6b40305d3a3', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000087.txt': '35f97313ac7cd6c26e2bf60e502610c70693975fadba2b31967eda87ec95792e', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000088.txt': 'd02e5bd9f0a740ba3b6b52ed0b960fddf650fb0893d326bb5530fed015e74b49', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000089.txt': '37c70aff6892aa5648ffd899685292e95d28ba75739ec70efed752aa261ab9d9', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000090.txt': '8320a63b9f9d80b97d4aa568529ce5f5fca7b80f4c3aa9f279645da7985e03b8', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000146.txt': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000147.txt': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'data/reviews/fronts-2026-11-26-r38/labels/val/new_000148.txt': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'data/reviews/fronts-2026-11-26-r38/manifest.json': '0921081930f49340ff34290d5752be2d669f084f4d1518e2019060a5600c43b5', 'docs/front-diagnosis-r39.md': '90a477377b22bfb302c9441d3622178f85c7764773d6cc21a330df806ff88ce2', 'results/evaluation/pr123-r37-20261126/comparison.json': 'b922b2a82552b652387c1d2f0caa024a0cc35a3aee5a6559f269149983580fec', 'results/evaluation/pr123-r37-20261126/predictions/candidate.jsonl': '7ec91a12a2882f5b641683963556f1e5e16148f376c118432b4d95bf1da493ef', 'results/evaluation/pr123-r39-diagnosis-20261126/clip-nms-probe.json': 'a297d37f6ddfc5670fb415cbc43bc22c1136e917854a0e878ad4396d67440d52', 'results/evaluation/pr123-r39-diagnosis-20261126/clip-nms-visual-review.json': '58b298edb35dce41cb99e2718a0f0ff3ef98e3439b19f57af38e932b32a42a45', 'results/evaluation/pr123-r39-diagnosis-20261126/clipped-nms-predictions.jsonl': 'd4b404997b891192839c6551fa4513fea49f2c51bbc7cfb9212185ace45ff0f2', 'results/evaluation/pr123-r39-diagnosis-20261126/counterfactual-predictions.jsonl': 'b575a2bc4d2009248db05db8570619913dfc3a495120187521b1e563cc77c717', 'results/evaluation/pr123-r39-diagnosis-20261126/diagnosis.json': 'b40017371a072498886e3f09f27d61fef090dd98ace0ee10fdbaa6eb44829a8e', 'results/evaluation/pr123-r39-diagnosis-20261126/duplicate-overlap-audit.json': 'e524ab2d9c81c6723830195ef9fdce02cd59b0b2f1b8d3867a64bb3efe2665b8', 'results/evaluation/pr123-r39-diagnosis-20261126/summary.json': 'ece57cf7f836a7d2fe476e5f552db3f896aaf31502c99b2dec8f1a50091e09ac', 'results/review/front-review-20261126-r38/row-maps.json': '56e7662a1facbeedcdfe29210c263c4bd1b75ce8b0149e5317138a942667dc9d', 'scripts/diagnose_pose_quality.py': 'a3baa1285d1580ee44547f7c3d606f524f2f40616fed568885746d2014159953', 'scripts/diagnose_r39_proposals.py': '6c695f20d116f984cbdc5df8052b25b888585d162d4f823d7b2d0feaee1f2b16', 'scripts/evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'scripts/pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4', 'scripts/pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'scripts/probe_r39_clipped_nms.py': 'eee866362423fef20ac892852d67591246044ef40b1daa9dae09e5874762959e'}
RESULTS = 'results/evaluation/pr123-r39-diagnosis-20261126'
WORK = Path(os.environ.get("BAOMAI_R39_WORK", str(Path("/content/baomai-r39") if Path("/content").is_dir() else Path.cwd() / "baomai-r39"))).expanduser().resolve()
SOURCE = WORK / "source"
SOURCE.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE = os.environ.get("BAOMAI_R39_SOURCE_DIR")

def acquire(item):
    relative, expected = item
    target = SOURCE / relative
    if target.exists():
        data = target.read_bytes()
        if hashlib.sha256(data).hexdigest() != expected:
            raise RuntimeError(f"Preserved changed file: {target}. Choose a new BAOMAI_R39_WORK folder.")
        return
    if LOCAL_SOURCE:
        data = (Path(LOCAL_SOURCE) / relative).read_bytes()
    else:
        url = f"https://raw.githubusercontent.com/Siuuuers/dl/{SOURCE_REVISION}/{relative}"
        with urllib.request.urlopen(url, timeout=45) as response:
            data = response.read()
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError(f"Pinned source hash differs: {relative}")
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".download")
    temporary.write_bytes(data)
    temporary.replace(target)

with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(acquire, FILES.items()))
print(f"Ready: {len(FILES)} pinned evidence/source files. CPU only; no ML installation.")


## Replay the evidence

Recompute baseline and confidence-probe scores from complete 20-image prediction caches, check the accepted labels and original R37 identities, and rebuild mechanism counts from the saved eligible raw proposals. Reapply the small clipped-box NMS function and reproduce both R38 and frozen666 comparisons. Both-corner success remains 570/583 within 5 pixels and 582/583 within 10 pixels on R38.

The underlying trace used fresh CPU inference and exactly reproduced all accepted baseline arrays. This notebook does not rerun the network. The compact evidence retains all qualifying proposals for the missed targets, but omits full dense tensors; independently establishing that an omitted tensor contains no other qualifying box requires the documented full trace command.


In [ ]:
for relative, expected in FILES.items():
    if hashlib.sha256((SOURCE / relative).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Evidence changed: {relative}")
sys.path.insert(0, str(SOURCE / "scripts"))
from diagnose_r39_proposals import replay_evidence
from probe_r39_clipped_nms import replay_probe

evidence_path = SOURCE / RESULTS / "diagnosis.json"
evidence = json.loads(evidence_path.read_text())
summary = replay_evidence(SOURCE, evidence_path)
expected_summary = json.loads((SOURCE / RESULTS / "summary.json").read_text())
if summary != expected_summary:
    raise RuntimeError("Replayed diagnosis differs from the saved result")
clip_summary = replay_probe(SOURCE, SOURCE / RESULTS / "clip-nms-probe.json")

def show_table(headers, rows):
    text = "| " + " | ".join(headers) + " |\n| " + " | ".join("---" for _ in headers) + " |\n"
    text += "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)
    try:
        from IPython.display import Markdown, display
        display(Markdown(text))
    except ImportError:
        print(text)

base, probe = summary["baseline"]["detection"], summary["counterfactual"]["detection"]
clipped = clip_summary["references"]["r38"]["probe"]["detection"]
show_table(["Accepted R38: 654 fronts", "Original 0.13", "Extra clipped NMS 0.70", "Exploratory conf 0.05"],
           [[label, base[key], clipped[key], probe[key]] for label, key in (
               ("Outputs", "prediction_count"), ("Matches", "tp"), ("Misses", "fn"),
               ("All unmatched", "fp"), ("Duplicate subset", "duplicate_fp"), ("Other unmatched", "other_fp"))])
mechanism_labels = {
    "no_iou_eligible_decoded_box": "Box overlap below 0.50",
    "confidence_gate": "Low confidence",
    "nms_suppression": "NMS suppression",
    "nms_or_output_limit_unresolved": "Unresolved filtering",
    "retained_assignment": "Matching assignment",
}
show_table(["Where the 11 baseline misses arise", "Count"],
           [[mechanism_labels[key], value] for key, value in summary["mechanism_counts"].items()])
show_table(["Image", "R38 row", "R37 row", "Mechanism", "Recovered at 0.05"],
           [[c["image_id"], c["r38_row"], c["r37_row"], mechanism_labels[c["mechanism"]],
             "Yes" if c["probe_recovers"] else "No"] for c in evidence["missed_cases"]])
print("All 20 baseline output arrays were reproduced exactly during the recorded fresh CPU trace.")
print("This notebook replays saved evidence and scores; it does not repeat that inference.")
print("The exploratory threshold is not adopted. All accepted labels and selected defaults are preserved.")
print("Extra clipped NMS removes nine outputs and preserves every existing matched prediction on R38 and frozen666.")
print(summary["replay_limit"])


## Export the next-step record

The ZIP includes the mechanism table, stable missed-front identities, both prediction/score evidence and the next-step guide. No labels, thresholds, model selection or training inputs are changed by running it. Use the guide to choose the next focused data or inference experiment.


In [ ]:
import csv, io, zipfile

rows = io.StringIO()
writer = csv.writer(rows)
writer.writerow(["image_id", "r38_row", "r37_row", "stable_id", "mechanism", "eligible_raw_boxes", "highest_eligible_confidence", "recovered_at_0_05"])
for case in evidence["missed_cases"]:
    eligible = case["iou_eligible_proposals"]
    writer.writerow([case["image_id"], case["r38_row"], case["r37_row"], case["stable_id"], case["mechanism"],
                     len(eligible), max((p["confidence"] for p in eligible), default=""), case["probe_recovers"]])
payloads = {
    "summary.json": (json.dumps(summary, indent=2, allow_nan=False) + "\n").encode(),
    "clip-nms-summary.json": (json.dumps(clip_summary, indent=2, allow_nan=False) + "\n").encode(),
    "baseline-predictions.jsonl": (SOURCE / "results/evaluation/pr123-r37-20261126/predictions/candidate.jsonl").read_bytes(),
    "missed-fronts.csv": rows.getvalue().encode(),
    "next-steps.md": (SOURCE / "docs/front-diagnosis-r39.md").read_bytes(),
    "source.json": (json.dumps({"revision": SOURCE_REVISION, "files_sha256": FILES,
                               "operation": "saved-evidence replay", "training": False,
                               "fresh_inference": False, "automatic_promotion": False}, indent=2) + "\n").encode(),
}
for name in ("diagnosis.json", "counterfactual-predictions.jsonl", "duplicate-overlap-audit.json",
             "clip-nms-probe.json", "clipped-nms-predictions.jsonl", "clip-nms-visual-review.json"):
    payloads[name] = (SOURCE / RESULTS / name).read_bytes()
ARCHIVE = WORK / "baomai-front-diagnosis-r39-results.zip"
temporary = ARCHIVE.with_suffix(".tmp")
with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for name, data in sorted(payloads.items()):
        item = zipfile.ZipInfo(name, date_time=(2026, 11, 26, 0, 0, 0))
        item.compress_type = zipfile.ZIP_DEFLATED
        bundle.writestr(item, data)
with zipfile.ZipFile(temporary) as bundle:
    if bundle.testzip() is not None:
        raise RuntimeError("Export archive failed its integrity check")
if ARCHIVE.exists() and ARCHIVE.read_bytes() != temporary.read_bytes():
    temporary.unlink()
    raise RuntimeError("Preserved a different existing archive. Choose a new BAOMAI_R39_WORK folder.")
temporary.replace(ARCHIVE)
print(f"Saved {ARCHIVE.name} ({ARCHIVE.stat().st_size:,} bytes). No new training is needed for this diagnosis.")
if os.environ.get("BAOMAI_R39_DOWNLOAD", "1") != "0":
    try:
        from google.colab import files
        files.download(str(ARCHIVE))
    except ImportError:
        try:
            from IPython.display import FileLink, display
            display(FileLink(str(ARCHIVE)))
        except ImportError:
            print(ARCHIVE)
